In [1]:
# using a hybrid approach combining both seq2seq and Xg_boost preictions

In [2]:
import pandas as pd
import numpy as np
import scipy.stats as stats

In [ ]:
def diebold_mariano_test(actual, pred1, pred2):
    e1 = np.abs(actual - pred1)
    e2 = np.abs(actual - pred2)
    d = e1 - e2
    mean_d = np.mean(d)
    var_d = np.var(d, ddof=1)
    if var_d == 0: 
        return 0.0, 1.0 
    dm_stat = mean_d / np.sqrt(var_d / len(d))
    p_value = 2 * (1 - stats.norm.cdf(np.abs(dm_stat)))
    return dm_stat, p_value

# Load the files
tft_df = pd.read_csv('tft_item_level (2).csv')
xgb_df = pd.read_csv('xgb_aggr.csv')

# Get the shared items
common_items = list(set(tft_df['item_id']) & set(xgb_df['item_id']))
xgb_winners = []

alpha = 0.05

print("Extracting XGBoost winners...")
for item in common_items:
    actual = tft_df[tft_df['item_id'] == item]['actual'].values
    tft_p = tft_df[tft_df['item_id'] == item]['tft_pred'].values
    xgb_p = xgb_df[xgb_df['item_id'] == item]['xgb_pred'].values
    
    dm_stat, p_val = diebold_mariano_test(actual, tft_p, xgb_p)
    
    # If p_val is significant AND dm_stat is positive (meaning TFT error was larger)
    if p_val < alpha and dm_stat > 0:
        xgb_winners.append(item)

print("\n" + "="*60)
print(f"🎯 FOUND {len(xgb_winners)} ITEMS WHERE XGBOOST WON")
print("="*60)
print("Copy and paste the code below directly into your Hybrid script:\n")
print(f"xgb_winners = {xgb_winners}")

Extracting XGBoost winners...

🎯 FOUND 17 ITEMS WHERE XGBOOST WON
Copy and paste the code below directly into your Hybrid script:

xgb_winners = ['FOODS_1_184', 'FOODS_1_181', 'FOODS_1_172', 'FOODS_1_032', 'FOODS_1_058', 'FOODS_1_183', 'FOODS_1_040', 'FOODS_1_110', 'FOODS_1_161', 'FOODS_1_043', 'FOODS_1_154', 'FOODS_1_085', 'FOODS_1_099', 'FOODS_1_044', 'FOODS_1_004', 'FOODS_1_162', 'FOODS_1_112']


In [4]:
print("Loading the best 2 models for forecasting...")
seq_df = pd.read_csv('seq2seq_item_level.csv')
xgb_df = pd.read_csv('xgb_aggr.csv') 

seq_df['date'] = pd.to_datetime(seq_df['date'])
xgb_df['date'] = pd.to_datetime(xgb_df['date'])

items = seq_df['item_id'].unique()

Loading the best 2 models for forecasting...


In [5]:
# selecting the best 17 items where xg_boost outperformed
xgb_winners = ['FOODS_1_154', 'FOODS_1_085', 'FOODS_1_058', 'FOODS_1_181', 'FOODS_1_183', 'FOODS_1_040', 'FOODS_1_161', 'FOODS_1_184', 'FOODS_1_110', 'FOODS_1_004', 'FOODS_1_032', 'FOODS_1_112', 'FOODS_1_044', 'FOODS_1_172', 'FOODS_1_043', 'FOODS_1_162', 'FOODS_1_099']


In [6]:
# constructing the hybrid engine
inventory_decisions = []
LEAD_TIME_DAYS = 7  # Delivery lag
SERVICE_LEVEL_Z = 1.65  # 95% confidence against stockouts

In [ ]:
# Mocking current stock for the 50 items
np.random.seed(42)
current_stock_levels = {item: np.random.randint(5, 60) for item in items}

print(f"Generating hybrid inventory strategy for {len(items)} items...")

for item in items:
    # --- MODEL ROUTING ---
    # We dynamically select the model based on the DM Test result
    if item in xgb_winners:
        item_data = xgb_df[xgb_df['item_id'] == item].copy()
        pred_col = 'xgb_pred'
        model_tag = 'XGBoost (Vol. Leader)'
    else:
        item_data = seq_df[seq_df['item_id'] == item].copy()
        pred_col = 'seq2seq_pred'
        model_tag = 'Seq2Seq (Reliability)'

    # Calculate Logic
    lead_time_demand = item_data.head(LEAD_TIME_DAYS)[pred_col].sum()
    
    sigma_demand = item_data[pred_col].std()
    safety_stock = SERVICE_LEVEL_Z * sigma_demand * np.sqrt(LEAD_TIME_DAYS)
    safety_stock = max(0, np.nan_to_num(safety_stock))

    rop = lead_time_demand + safety_stock
    current_inv = current_stock_levels[item]
    
    if current_inv <= rop:
        status = "REORDER"
        cycle_demand = item_data.head(14)[pred_col].sum()
        order_qty = max(0, cycle_demand + safety_stock - current_inv)
    else:
        status = "STABLE"
        order_qty = 0

    # APPEND TO LIST 
    inventory_decisions.append({
        'item_id': item,
        'selected_model': model_tag,
        'current_stock': current_inv,
        'reorder_point': round(rop, 1),
        'safety_stock_buffer': round(safety_stock, 1),
        'decision': status,
        'suggested_order': round(order_qty, 0)
    })

Generating hybrid inventory strategy for 50 items...


In [ ]:
# 4. EXPORT FINAL ACTION REPORT

report = pd.DataFrame(inventory_decisions)
report.to_csv('hybrid_inventory_action_report.csv', index=False)

print("\n" + "="*80)
print("📦 STOCKCAST HYBRID INVENTORY STATUS")
print("="*80)
print(report.sort_values('decision', ascending=False).head(10).to_string(index=False))
print("="*80)
print(f"\nFinal report:'hybrid_inventory_action_report.csv'")